Установка и импорты

In [2]:
# Установка зависимостей
!pip install -q langchain langchain-openai langchain-community sentence-transformers faiss-cpu python-dotenv

import os
from google.colab import userdata
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.runnables import RunnablePassthrough

print("✅ Все библиотеки загружены")

✅ Все библиотеки загружены


Создаём тестовую базу знаний

In [4]:
# Тестовая база знаний (потом заменишь на свои данные)
knowledge_text = """
Возврат товара:
Для возврата товара необходимо иметь при себе чек и паспорт.
Срок возврата составляет 14 дней с момента покупки.
Товар должен быть в оригинальной упаковке без следов использования.
Деньги возвращаются на карту в течение 10 рабочих дней.

Доставка:
Доставка осуществляется курьерской службой по всей России.
По Москве доставка занимает 1-2 дня и стоит 350 рублей.
По России доставка занимает 5-10 дней и стоит от 500 рублей.
Бесплатная доставка при заказе от 5000 рублей.

График работы:
Служба поддержки работает с понедельника по пятницу с 9:00 до 21:00.
В субботу и воскресенье с 10:00 до 18:00.
Телефон горячей линии: 8-800-555-35-35.
Email: support@company.ru.

Гарантия:
Гарантийный срок на все товары составляет 1 год с момента покупки.
Гарантийный ремонт осуществляется в авторизованных сервисных центрах.
Для гарантийного ремонта необходим гарантийный талон и чек.
Гарантия не распространяется на механические повреждения.

Приложение:
Мобильное приложение доступно в App Store и Google Play.
Для входа используйте номер телефона или email.
При первом входе необходимо подтвердить номер через SMS-код.
В приложении можно отслеживать статус заказа и историю покупок.
"""

print("✅ База знаний создана")
print(f"📊 Размер текста: {len(knowledge_text)} символов")

✅ База знаний создана
📊 Размер текста: 1193 символов


Чанкинг — разбиваем текст на части

In [5]:
# Создаём сплиттер
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,        # размер чанка в символах
    chunk_overlap=50,      # перекрытие между чанками
    separators=["\n\n", "\n", ". ", " ", ""]  # приоритет разделителей
)

# Разбиваем текст на чанки
chunks = text_splitter.split_text(knowledge_text)

print(f"✅ Текст разбит на {len(chunks)} чанков\n")
for i, chunk in enumerate(chunks):
    print(f"--- Чанк {i+1} ({len(chunk)} символов) ---")
    print(chunk[:150] + "...")
    print()

✅ Текст разбит на 5 чанков

--- Чанк 1 (254 символов) ---
Возврат товара:
Для возврата товара необходимо иметь при себе чек и паспорт. 
Срок возврата составляет 14 дней с момента покупки. 
Товар должен быть в...

--- Чанк 2 (232 символов) ---
Доставка:
Доставка осуществляется курьерской службой по всей России.
По Москве доставка занимает 1-2 дня и стоит 350 рублей.
По России доставка занима...

--- Чанк 3 (192 символов) ---
График работы:
Служба поддержки работает с понедельника по пятницу с 9:00 до 21:00.
В субботу и воскресенье с 10:00 до 18:00.
Телефон горячей линии: 8...

--- Чанк 4 (264 символов) ---
Гарантия:
Гарантийный срок на все товары составляет 1 год с момента покупки.
Гарантийный ремонт осуществляется в авторизованных сервисных центрах.
Для...

--- Чанк 5 (241 символов) ---
Приложение:
Мобильное приложение доступно в App Store и Google Play.
Для входа используйте номер телефона или email.
При первом входе необходимо подтв...



Создаём векторное хранилище

In [6]:
# Создаём эмбеддер
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=userdata.get('OPENAI_API_KEY')
)

# Создаём векторное хранилище
vector_store = FAISS.from_texts(chunks, embeddings)

print(f"✅ Векторное хранилище создано")
print(f"📐 Размерность эмбеддингов: 1536")
print(f"📊 Чанков в хранилище: {vector_store.index.ntotal}")

✅ Векторное хранилище создано
📐 Размерность эмбеддингов: 1536
📊 Чанков в хранилище: 5


Создаём retriever

In [7]:
# Создаём retriever — он будет искать релевантные чанки
retriever = vector_store.as_retriever(
    search_type="similarity",  # поиск по косинусному сходству
    search_kwargs={"k": 3}     # возвращаем топ-3 чанка
)

# Тестируем
test_query = "Как вернуть товар?"
docs = retriever.invoke(test_query)

print(f"🔍 Запрос: '{test_query}'\n")
for i, doc in enumerate(docs):
    print(f"--- Чанк {i+1} ---")
    print(doc.page_content[:200])
    print()

🔍 Запрос: 'Как вернуть товар?'

--- Чанк 1 ---
Возврат товара:
Для возврата товара необходимо иметь при себе чек и паспорт. 
Срок возврата составляет 14 дней с момента покупки. 
Товар должен быть в оригинальной упаковке без следов использования.
Д

--- Чанк 2 ---
Гарантия:
Гарантийный срок на все товары составляет 1 год с момента покупки.
Гарантийный ремонт осуществляется в авторизованных сервисных центрах.
Для гарантийного ремонта необходим гарантийный талон 

--- Чанк 3 ---
Приложение:
Мобильное приложение доступно в App Store и Google Play.
Для входа используйте номер телефона или email.
При первом входе необходимо подтвердить номер через SMS-код.
В приложении можно отс



Создаём LLM и промпт

In [8]:
# Создаём LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    api_key=userdata.get('OPENAI_API_KEY')
)

# Создаём промпт-шаблон
prompt = ChatPromptTemplate.from_messages([
    ("system", """Ты — полезный ассистент службы поддержки.
Отвечай на вопросы пользователей, используя ТОЛЬКО предоставленный контекст.
Если ответа нет в контексте — честно скажи об этом, не придумывай информацию.
Отвечай кратко и по делу."""),
    ("human", """Контекст:
{context}

Вопрос пользователя: {question}

Ответ:""")
])

print("✅ LLM и промпт созданы")

✅ LLM и промпт созданы


Собираем RAG-цепочку

In [9]:
def format_docs(docs):
    """Форматирует найденные документы в строку"""
    return "\n\n".join(doc.page_content for doc in docs)

# Собираем цепочку через LCEL
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG-цепочка собрана!")
print("\n📋 Схема цепочки:")
print("  query → retriever → format_docs → prompt → LLM → ответ")

✅ RAG-цепочка собрана!

📋 Схема цепочки:
  query → retriever → format_docs → prompt → LLM → ответ


Тестируем

In [10]:
# Тестовые вопросы
test_questions = [
    "Как мне вернуть товар?",
    "Сколько стоит доставка в Казань?",
    "Какой график работы поддержки?",
    "Что делать, если товар сломался через полгода?",
    "Можно ли заказать пиццу?"
]

print("🧪 ТЕСТИРОВАНИЕ RAG-АГЕНТА")
print("="*50)

for question in test_questions:
    print(f"\n❓ Вопрос: {question}")
    print("-"*30)

    # Получаем ответ
    answer = rag_chain.invoke(question)
    print(f"🤖 Ответ: {answer}")

    # Показываем найденные чанки
    docs = retriever.invoke(question)
    print(f"📚 Найдено чанков: {len(docs)}")
    for i, doc in enumerate(docs):
        print(f"   Чанк {i+1}: {doc.page_content[:100]}...")
    print()

🧪 ТЕСТИРОВАНИЕ RAG-АГЕНТА

❓ Вопрос: Как мне вернуть товар?
------------------------------
🤖 Ответ: Для возврата товара необходимо иметь при себе чек и паспорт. Срок возврата составляет 14 дней с момента покупки, и товар должен быть в оригинальной упаковке без следов использования. Деньги возвращаются на карту в течение 10 рабочих дней.
📚 Найдено чанков: 3
   Чанк 1: Возврат товара:
Для возврата товара необходимо иметь при себе чек и паспорт. 
Срок возврата составля...
   Чанк 2: Гарантия:
Гарантийный срок на все товары составляет 1 год с момента покупки.
Гарантийный ремонт осущ...
   Чанк 3: Приложение:
Мобильное приложение доступно в App Store и Google Play.
Для входа используйте номер тел...


❓ Вопрос: Сколько стоит доставка в Казань?
------------------------------
🤖 Ответ: Доставка в Казань стоит от 500 рублей и занимает 5-10 дней.
📚 Найдено чанков: 3
   Чанк 1: Доставка:
Доставка осуществляется курьерской службой по всей России.
По Москве доставка занимает 1-2...
   Чанк 2: Возвр

Добавляем память — агент помнит контекст диалога

In [11]:
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import MessagesPlaceholder

# Промпт с историей диалога
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", """Ты — полезный ассистент службы поддержки.
Отвечай на вопросы, используя ТОЛЬКО контекст из базы знаний.
Если ответа нет — честно скажи."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", """Контекст из базы знаний:
{context}

Вопрос: {question}""")
])

# Цепочка с историей
from langchain_core.runnables.history import RunnableWithMessageHistory

chat_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
        "chat_history": lambda x: x  # будет заполняться автоматически
    }
    | chat_prompt
    | llm
    | StrOutputParser()
)

print("✅ Чат-цепочка с памятью создана")
print("ℹ️ Для демо используем упрощённую версию с ручным управлением историей")

✅ Чат-цепочка с памятью создана
ℹ️ Для демо используем упрощённую версию с ручным управлением историей


In [12]:
# Демо диалога с ручным управлением историей
print("💬 ДИАЛОГ С ПАМЯТЬЮ")
print("="*50)

chat_history = []

questions = [
    "Как вернуть товар?",
    "А сколько дней займёт возврат денег?",  # уточняющий вопрос — агент должен помнить контекст
    "Спасибо! А какой у вас график работы?"
]

for question in questions:
    # Получаем контекст
    docs = retriever.invoke(question)
    context = format_docs(docs)

    # Формируем сообщения
    messages = [{"role": "system", "content": "Ты — ассистент поддержки. Отвечай по контексту."}]

    # Добавляем историю
    for msg in chat_history:
        messages.append(msg)

    # Добавляем текущий вопрос
    messages.append({"role": "user", "content": f"Контекст:\n{context}\n\nВопрос: {question}"})

    # Генерируем ответ
    response = llm.invoke(
        [{"role": m["role"], "content": m["content"]} for m in messages]
    )
    answer = response.content

    print(f"\n❓ {question}")
    print(f"🤖 {answer}")

    # Обновляем историю
    chat_history.append({"role": "user", "content": question})
    chat_history.append({"role": "assistant", "content": answer})

print("\n📝 Итоговая история диалога:")
for msg in chat_history:
    role = "👤" if msg["role"] == "user" else "🤖"
    print(f"{role} {msg['content'][:100]}...")

💬 ДИАЛОГ С ПАМЯТЬЮ

❓ Как вернуть товар?
🤖 Для возврата товара вам необходимо выполнить следующие шаги:

1. Убедитесь, что у вас есть чек и паспорт.
2. Проверьте, что товар находится в оригинальной упаковке и не имеет следов использования.
3. Верните товар в течение 14 дней с момента покупки.
4. После возврата деньги будут возвращены на вашу карту в течение 10 рабочих дней. 

Если у вас есть дополнительные вопросы, не стесняйтесь спрашивать!

❓ А сколько дней займёт возврат денег?
🤖 Возврат денег займет до 10 рабочих дней после возврата товара.

❓ Спасибо! А какой у вас график работы?
🤖 Служба поддержки работает с понедельника по пятницу с 9:00 до 21:00, а в субботу и воскресенье — с 10:00 до 18:00.

📝 Итоговая история диалога:
👤 Как вернуть товар?...
🤖 Для возврата товара вам необходимо выполнить следующие шаги:

1. Убедитесь, что у вас есть чек и пас...
👤 А сколько дней займёт возврат денег?...
🤖 Возврат денег займет до 10 рабочих дней после возврата товара....
👤 Спасибо! А какой у в

Оценка качества

In [13]:
# Создаём golden set
eval_set = [
    {"question": "Как вернуть товар?", "expected_keywords": ["чек", "паспорт", "14 дней"]},
    {"question": "Сколько стоит доставка?", "expected_keywords": ["350", "500", "курьер"]},
    {"question": "Какой график работы?", "expected_keywords": ["9:00", "21:00", "понедельник"]},
    {"question": "Как скачать приложение?", "expected_keywords": ["App Store", "Google Play", "SMS"]},
    {"question": "Какой срок гарантии?", "expected_keywords": ["1 год", "гарантийный талон"]},
]

print("📊 ОЦЕНКА КАЧЕСТВА RAG-АГЕНТА")
print("="*50)

score = 0
for item in eval_set:
    question = item["question"]
    expected = item["expected_keywords"]

    answer = rag_chain.invoke(question).lower()

    # Проверяем наличие ключевых слов
    found = [kw for kw in expected if kw.lower() in answer]
    item_score = len(found) / len(expected)
    score += item_score

    status = "✅" if item_score >= 0.66 else "⚠️" if item_score > 0 else "❌"
    print(f"{status} {question}")
    print(f"   Найдено: {found}/{expected}")
    print(f"   Ответ: {answer[:100]}...")
    print()

total_score = score / len(eval_set)
print(f"📈 Итоговая оценка: {total_score:.2f} ({total_score*100:.0f}%)")

📊 ОЦЕНКА КАЧЕСТВА RAG-АГЕНТА
✅ Как вернуть товар?
   Найдено: ['чек', 'паспорт', '14 дней']/['чек', 'паспорт', '14 дней']
   Ответ: для возврата товара необходимо иметь при себе чек и паспорт. срок возврата составляет 14 дней с моме...

✅ Сколько стоит доставка?
   Найдено: ['350', '500']/['350', '500', 'курьер']
   Ответ: доставка по москве стоит 350 рублей и занимает 1-2 дня. по россии доставка стоит от 500 рублей и зан...

✅ Какой график работы?
   Найдено: ['9:00', '21:00', 'понедельник']/['9:00', '21:00', 'понедельник']
   Ответ: служба поддержки работает с понедельника по пятницу с 9:00 до 21:00, а в субботу и воскресенье с 10:...

✅ Как скачать приложение?
   Найдено: ['App Store', 'Google Play']/['App Store', 'Google Play', 'SMS']
   Ответ: скачайте приложение из app store или google play....

⚠️ Какой срок гарантии?
   Найдено: ['1 год']/['1 год', 'гарантийный талон']
   Ответ: гарантийный срок на все товары составляет 1 год с момента покупки....

📈 Итоговая оценка: 0.77 (77%)

Установка реранкера

In [14]:
# Установка cross-encoder
!pip install -q sentence-transformers

from sentence_transformers import CrossEncoder

print("✅ Sentence-Transformers установлен")

✅ Sentence-Transformers установлен


Создаём реранкер

In [15]:
# Загружаем cross-encoder
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    max_length=512  # макс длина текста
)

print("✅ Рера́нкер загружен")
print(f"🧠 Модель: cross-encoder/ms-marco-MiniLM-L-6-v2")
print(f"📏 Max length: 512 токенов")
print(f"⚡ Тип: cross-encoder (видит запрос и чанк вместе)")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Рера́нкер загружен
🧠 Модель: cross-encoder/ms-marco-MiniLM-L-6-v2
📏 Max length: 512 токенов
⚡ Тип: cross-encoder (видит запрос и чанк вместе)


Функция retrieval с реранкером

In [16]:
def retrieve_with_rerank(query, k_retrieve=6, k_final=3):
    """
    Retrieval с реранкером:
    1. Достаём k_retrieve кандидатов из FAISS
    2. Реранкер переоценивает их
    3. Возвращаем топ-k_final
    """
    # Шаг 1: Получаем кандидатов из FAISS
    candidates = retriever.invoke(query)
    # Берём больше, чем нужно для финала
    candidates = candidates[:k_retrieve]

    # Шаг 2: Реранкер оценивает пары (query, candidate)
    pairs = [[query, doc.page_content] for doc in candidates]
    scores = reranker.predict(pairs)

    # Шаг 3: Сортируем по score реранкера
    ranked = sorted(
        zip(candidates, scores),
        key=lambda x: x[1],
        reverse=True
    )

    # Шаг 4: Возвращаем топ-k_final
    return [doc for doc, score in ranked[:k_final]]

# Тестируем
test_query = "Как вернуть товар?"
docs_with_rerank = retrieve_with_rerank(test_query)

print(f"🔍 Запрос: '{test_query}'\n")
print("📊 С реранкером (топ-3):\n")
for i, doc in enumerate(docs_with_rerank, 1):
    print(f"--- Чанк {i} ---")
    print(doc.page_content[:150])
    print()

🔍 Запрос: 'Как вернуть товар?'

📊 С реранкером (топ-3):

--- Чанк 1 ---
Возврат товара:
Для возврата товара необходимо иметь при себе чек и паспорт. 
Срок возврата составляет 14 дней с момента покупки. 
Товар должен быть в

--- Чанк 2 ---
Гарантия:
Гарантийный срок на все товары составляет 1 год с момента покупки.
Гарантийный ремонт осуществляется в авторизованных сервисных центрах.
Для

--- Чанк 3 ---
Приложение:
Мобильное приложение доступно в App Store и Google Play.
Для входа используйте номер телефона или email.
При первом входе необходимо подтв



Сравнение с/без реранкера

In [17]:
# Сравниваем retrieval с реранкером и без
test_queries = [
    "Как вернуть товар?",
    "Какая гарантия на товары?",
    "Как работает доставка?"
]

print("🔄 СРАВНЕНИЕ RETRIEVAL С РЕРАНКЕРОМ И БЕЗ")
print("="*60)

for query in test_queries:
    print(f"\n🔍 Запрос: '{query}'")
    print("-"*40)

    # Без реранкера
    docs_simple = retriever.invoke(query)[:3]
    print("БЕЗ реранкера (FAISS top-3):")
    for i, doc in enumerate(docs_simple, 1):
        print(f"  {i}. {doc.page_content[:80]}...")

    # С реранкером
    docs_rerank = retrieve_with_rerank(query)
    print("С реранкером (top-3 из 6):")
    for i, doc in enumerate(docs_rerank, 1):
        print(f"  {i}. {doc.page_content[:80]}...")
    print()

🔄 СРАВНЕНИЕ RETRIEVAL С РЕРАНКЕРОМ И БЕЗ

🔍 Запрос: 'Как вернуть товар?'
----------------------------------------
БЕЗ реранкера (FAISS top-3):
  1. Возврат товара:
Для возврата товара необходимо иметь при себе чек и паспорт. 
Ср...
  2. Гарантия:
Гарантийный срок на все товары составляет 1 год с момента покупки.
Гар...
  3. Приложение:
Мобильное приложение доступно в App Store и Google Play.
Для входа и...
С реранкером (top-3 из 6):
  1. Возврат товара:
Для возврата товара необходимо иметь при себе чек и паспорт. 
Ср...
  2. Гарантия:
Гарантийный срок на все товары составляет 1 год с момента покупки.
Гар...
  3. Приложение:
Мобильное приложение доступно в App Store и Google Play.
Для входа и...


🔍 Запрос: 'Какая гарантия на товары?'
----------------------------------------
БЕЗ реранкера (FAISS top-3):
  1. Гарантия:
Гарантийный срок на все товары составляет 1 год с момента покупки.
Гар...
  2. Возврат товара:
Для возврата товара необходимо иметь при себе чек и паспорт. 
Ср...
  3. Дос

Обновлённая RAG-цепочка с реранкером

In [18]:
# Обновлённая цепочка с реранкером
def format_docs_with_rerank(query):
    """Retrieval с реранкером → форматирование"""
    docs = retrieve_with_rerank(query)
    return format_docs(docs)

rag_chain_rerank = (
    {
        "context": lambda q: format_docs_with_rerank(q),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG-цепочка с реранкером собрана!")
print("\n📋 Схема:")
print("  query → FAISS (топ-6) → Cross-Encoder → топ-3 → prompt → LLM → ответ")

✅ RAG-цепочка с реранкером собрана!

📋 Схема:
  query → FAISS (топ-6) → Cross-Encoder → топ-3 → prompt → LLM → ответ


Тестируем улучшенную версию

In [19]:
# Тестируем улучшенного агента
print("🧪 ТЕСТИРОВАНИЕ RAG-АГЕНТА С РЕРАНКЕРОМ")
print("="*50)

for question in test_questions:
    print(f"\n❓ Вопрос: {question}")
    print("-"*30)

    answer = rag_chain_rerank.invoke(question)
    print(f"🤖 Ответ: {answer}")
    print()

🧪 ТЕСТИРОВАНИЕ RAG-АГЕНТА С РЕРАНКЕРОМ

❓ Вопрос: Как мне вернуть товар?
------------------------------
🤖 Ответ: Для возврата товара необходимо иметь при себе чек и паспорт. Срок возврата составляет 14 дней с момента покупки, и товар должен быть в оригинальной упаковке без следов использования. Деньги возвращаются на карту в течение 10 рабочих дней.


❓ Вопрос: Сколько стоит доставка в Казань?
------------------------------
🤖 Ответ: Доставка в Казань стоит от 500 рублей и занимает 5-10 дней.


❓ Вопрос: Какой график работы поддержки?
------------------------------
🤖 Ответ: Служба поддержки работает с понедельника по пятницу с 9:00 до 21:00, а в субботу и воскресенье с 10:00 до 18:00.


❓ Вопрос: Что делать, если товар сломался через полгода?
------------------------------
🤖 Ответ: Если товар сломался через полгода, вы можете обратиться в авторизованный сервисный центр для гарантийного ремонта. Для этого вам понадобятся гарантийный талон и чек. Обратите внимание, что гарантия не распрос

Создаём sparse-индекс (TF-IDF)

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Создаём TF-IDF векторизатор
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(chunks)  # преобразуем все чанки

print("✅ Sparse-индекс создан")
print(f"📊 Словарь: {len(tfidf.vocabulary_)} уникальных слов")
print(f"📐 Размер матрицы: {tfidf_matrix.shape[0]} чанков × {tfidf_matrix.shape[1]} признаков")

✅ Sparse-индекс создан
📊 Словарь: 114 уникальных слов
📐 Размер матрицы: 5 чанков × 114 признаков


Функция sparse-поиска

In [24]:
from sklearn.metrics.pairwise import cosine_similarity

def sparse_search(query, top_k=6):
    """Поиск по ключевым словам через TF-IDF"""
    query_vec = tfidf.transform([query])
    scores = cosine_similarity(query_vec, tfidf_matrix)[0]

    # Индексы топ-K чанков
    top_indices = scores.argsort()[-top_k:][::-1]

    results = []
    for idx in top_indices:
        if scores[idx] > 0:  # только если есть совпадения
            results.append({
                "content": chunks[idx],
                "score": float(scores[idx])
            })

    return results

# Тестируем
print("🔍 Sparse-поиск: 'курьер доставка'\n")
for r in sparse_search("курьер доставка", top_k=3):
    print(f"  Score: {r['score']:.3f} | {r['content'][:80]}...")

🔍 Sparse-поиск: 'курьер доставка'

  Score: 0.606 | Доставка:
Доставка осуществляется курьерской службой по всей России.
По Москве д...


Гибридный поиск (dense + sparse)

In [25]:
def hybrid_search(query, k_retrieve=6, k_final=3, dense_weight=0.7):
    """
    Гибридный поиск:
    1. Dense (FAISS) — семантический
    2. Sparse (TF-IDF) — ключевые слова
    3. Объединение с весами
    """

    # --- Dense (FAISS) ---
    candidates_dense = retriever.invoke(query)[:k_retrieve]

    # Нормализуем dense-скоры (приводим к [0, 1])
    dense_scores = {}
    # Получаем скоры из FAISS (косинусное сходство)
    query_vec = embeddings.embed_query(query)
    faiss_scores = []
    for doc in candidates_dense:
        doc_vec = embeddings.embed_query(doc.page_content)
        # Косинусное сходство через скалярное произведение (векторы нормализованы)
        score = sum(a * b for a, b in zip(query_vec, doc_vec))
        faiss_scores.append(score)

    max_dense = max(faiss_scores) if faiss_scores else 1.0
    for doc, score in zip(candidates_dense, faiss_scores):
        dense_scores[doc.page_content] = score / max_dense if max_dense > 0 else 0

    # --- Sparse (TF-IDF) ---
    sparse_results = sparse_search(query, top_k=k_retrieve)
    max_sparse = max([r["score"] for r in sparse_results]) if sparse_results else 1.0
    sparse_scores = {}
    for r in sparse_results:
        sparse_scores[r["content"]] = r["score"] / max_sparse if max_sparse > 0 else 0

    # --- Объединение ---
    combined = {}
    all_contents = list(set([doc.page_content for doc in candidates_dense] +
                            [r["content"] for r in sparse_results]))

    for content in all_contents:
        dense = dense_scores.get(content, 0)
        sparse = sparse_scores.get(content, 0)
        combined[content] = dense_weight * dense + (1 - dense_weight) * sparse

    # Сортируем
    sorted_contents = sorted(combined, key=combined.get, reverse=True)[:k_final]

    # Формируем результат (как объекты Document)
    result_docs = []
    for content in sorted_contents:
        for doc in candidates_dense:
            if doc.page_content == content:
                result_docs.append(doc)
                break
        else:
            # Если чанк пришёл только из sparse
            from langchain_core.documents import Document
            result_docs.append(Document(page_content=content))

    return result_docs

# Тестируем
print("🔍 Гибридный поиск: 'Как вернуть товар?'\n")
docs_hybrid = hybrid_search("Как вернуть товар?")
for i, doc in enumerate(docs_hybrid, 1):
    print(f"--- Чанк {i} ---")
    print(doc.page_content[:120])
    print()

🔍 Гибридный поиск: 'Как вернуть товар?'

--- Чанк 1 ---
Возврат товара:
Для возврата товара необходимо иметь при себе чек и паспорт. 
Срок возврата составляет 14 дней с момента

--- Чанк 2 ---
Гарантия:
Гарантийный срок на все товары составляет 1 год с момента покупки.
Гарантийный ремонт осуществляется в авториз

--- Чанк 3 ---
Приложение:
Мобильное приложение доступно в App Store и Google Play.
Для входа используйте номер телефона или email.
При



RAG-цепочка с гибридным поиском

In [26]:
# Цепочка с гибридным поиском
rag_chain_hybrid = (
    {
        "context": lambda q: format_docs(hybrid_search(q)),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG-цепочка с гибридным поиском собрана!")
print("\n📋 Схема:")
print("  query → FAISS (dense) + TF-IDF (sparse) → объединение → топ-3 → prompt → LLM")

# Тестируем
print("\n" + "="*50)
print("🧪 ТЕСТ: Гибридный поиск")
print("="*50)

for question in test_questions:
    print(f"\n❓ {question}")
    answer = rag_chain_hybrid.invoke(question)
    print(f"🤖 {answer}")
    print()

✅ RAG-цепочка с гибридным поиском собрана!

📋 Схема:
  query → FAISS (dense) + TF-IDF (sparse) → объединение → топ-3 → prompt → LLM

🧪 ТЕСТ: Гибридный поиск

❓ Как мне вернуть товар?
🤖 Для возврата товара необходимо иметь при себе чек и паспорт. Срок возврата составляет 14 дней с момента покупки, и товар должен быть в оригинальной упаковке без следов использования. Деньги возвращаются на карту в течение 10 рабочих дней.


❓ Сколько стоит доставка в Казань?
🤖 Доставка в Казань стоит от 500 рублей и занимает 5-10 дней.


❓ Какой график работы поддержки?
🤖 Служба поддержки работает с понедельника по пятницу с 9:00 до 21:00, а в субботу и воскресенье с 10:00 до 18:00.


❓ Что делать, если товар сломался через полгода?
🤖 Если товар сломался через полгода, вы можете обратиться в авторизованный сервисный центр для гарантийного ремонта. Для этого вам понадобятся гарантийный талон и чек. Помните, что гарантия не распространяется на механические повреждения.


❓ Можно ли заказать пиццу?
🤖 В конт

Функция с мониторингом latency

In [27]:
import time

def search_with_monitoring(query, search_type="simple"):
    """
    Поиск с замером latency на каждом этапе.
    search_type: "simple", "rerank", "hybrid"
    """
    monitoring = {"query": query, "search_type": search_type}

    # 1. Retrieval
    t0 = time.time()
    if search_type == "simple":
        docs = retriever.invoke(query)[:3]
    elif search_type == "rerank":
        docs = retrieve_with_rerank(query)
    elif search_type == "hybrid":
        docs = hybrid_search(query)
    else:
        docs = retriever.invoke(query)[:3]
    monitoring["retrieval_ms"] = round((time.time() - t0) * 1000, 2)
    monitoring["chunks_found"] = len(docs)

    # 2. Форматирование контекста
    t0 = time.time()
    context = format_docs(docs)
    monitoring["format_ms"] = round((time.time() - t0) * 1000, 2)
    monitoring["context_length"] = len(context)

    # 3. LLM-генерация
    t0 = time.time()
    if search_type == "simple":
        answer = rag_chain.invoke(query)
    elif search_type == "rerank":
        answer = rag_chain_rerank.invoke(query)
    elif search_type == "hybrid":
        answer = rag_chain_hybrid.invoke(query)
    else:
        answer = rag_chain.invoke(query)
    monitoring["generation_ms"] = round((time.time() - t0) * 1000, 2)
    monitoring["answer_length"] = len(answer)

    # 4. Итого
    monitoring["total_ms"] = round(
        monitoring["retrieval_ms"] + monitoring["format_ms"] + monitoring["generation_ms"], 2
    )

    return answer, docs, monitoring

Тестируем все три варианта

In [28]:
# Сравниваем latency всех трёх подходов
test_query = "Как вернуть товар?"

print("⏱️ СРАВНЕНИЕ LATENCY ТРЁХ ПОДХОДОВ")
print("="*60)

for search_type in ["simple", "rerank", "hybrid"]:
    answer, docs, mon = search_with_monitoring(test_query, search_type)

    print(f"\n📊 {search_type.upper()}")
    print("-"*40)
    print(f"  🔍 Retrieval:  {mon['retrieval_ms']} мс ({mon['chunks_found']} чанков)")
    print(f"  📝 Format:     {mon['format_ms']} мс")
    print(f"  🤖 Generation: {mon['generation_ms']} мс")
    print(f"  ⏱️ Total:      {mon['total_ms']} мс")
    print(f"  💬 Ответ:      {answer[:100]}...")

⏱️ СРАВНЕНИЕ LATENCY ТРЁХ ПОДХОДОВ

📊 SIMPLE
----------------------------------------
  🔍 Retrieval:  321.11 мс (3 чанков)
  📝 Format:     0.01 мс
  🤖 Generation: 1201.29 мс
  ⏱️ Total:      1522.41 мс
  💬 Ответ:      Для возврата товара необходимо иметь при себе чек и паспорт. Срок возврата составляет 14 дней с моме...

📊 RERANK
----------------------------------------
  🔍 Retrieval:  499.27 мс (3 чанков)
  📝 Format:     0.01 мс
  🤖 Generation: 1742.53 мс
  ⏱️ Total:      2241.81 мс
  💬 Ответ:      Для возврата товара необходимо иметь при себе чек и паспорт. Срок возврата составляет 14 дней с моме...

📊 HYBRID
----------------------------------------
  🔍 Retrieval:  811.3 мс (3 чанков)
  📝 Format:     0.01 мс
  🤖 Generation: 2002.25 мс
  ⏱️ Total:      2813.56 мс
  💬 Ответ:      Для возврата товара необходимо иметь при себе чек и паспорт. Срок возврата составляет 14 дней с моме...


Structured logging

In [29]:
from datetime import datetime
import json

class RAGLogger:
    """Простой structured logger для RAG-пайплайна"""

    def __init__(self):
        self.logs = []

    def log(self, event, **kwargs):
        """Записывает событие с метаданными"""
        entry = {
            "timestamp": datetime.now().isoformat(),
            "event": event,
            **kwargs
        }
        self.logs.append(entry)
        # Красивый вывод в консоль
        print(f"[{entry['timestamp']}] {event}: {json.dumps(kwargs, ensure_ascii=False, indent=2)}")

    def get_stats(self):
        """Возвращает статистику по логам"""
        if not self.logs:
            return "Нет логов"

        total_requests = len([l for l in self.logs if l["event"] == "request_completed"])
        avg_latency = sum(l["total_ms"] for l in self.logs if l["event"] == "request_completed") / total_requests if total_requests > 0 else 0

        return {
            "total_requests": total_requests,
            "avg_latency_ms": round(avg_latency, 2),
            "events": len(self.logs)
        }

# Создаём логгер
logger = RAGLogger()
print("✅ RAG Logger создан")

✅ RAG Logger создан


RAG-пайплайн с логированием

In [30]:
def rag_with_logging(query, search_type="simple"):
    """RAG-пайплайн с полным логированием"""

    logger.log("request_started", query=query, search_type=search_type)

    # 1. Retrieval
    t0 = time.time()
    if search_type == "simple":
        docs = retriever.invoke(query)[:3]
    elif search_type == "rerank":
        docs = retrieve_with_rerank(query)
    elif search_type == "hybrid":
        docs = hybrid_search(query)
    else:
        docs = retriever.invoke(query)[:3]
    retrieval_ms = round((time.time() - t0) * 1000, 2)

    logger.log("retrieval_completed",
               chunks_found=len(docs),
               latency_ms=retrieval_ms,
               top_chunk=docs[0].page_content[:50] + "..." if docs else "none")

    # 2. Генерация
    t0 = time.time()
    if search_type == "simple":
        answer = rag_chain.invoke(query)
    elif search_type == "rerank":
        answer = rag_chain_rerank.invoke(query)
    elif search_type == "hybrid":
        answer = rag_chain_hybrid.invoke(query)
    else:
        answer = rag_chain.invoke(query)
    generation_ms = round((time.time() - t0) * 1000, 2)

    total_ms = retrieval_ms + generation_ms

    logger.log("request_completed",
               answer_preview=answer[:100] + "...",
               answer_length=len(answer),
               retrieval_ms=retrieval_ms,
               generation_ms=generation_ms,
               total_ms=total_ms)

    return answer, docs

# Тестируем
print("🧪 ТЕСТ RAG С ЛОГИРОВАНИЕМ")
print("="*50)

for question in test_questions[:3]:
    print(f"\n❓ {question}")
    answer, docs = rag_with_logging(question, "hybrid")
    print(f"🤖 {answer[:100]}...\n")

# Статистика
print("\n" + "="*50)
print("📊 СТАТИСТИКА")
print("="*50)
stats = logger.get_stats()
for k, v in stats.items():
    print(f"  {k}: {v}")

🧪 ТЕСТ RAG С ЛОГИРОВАНИЕМ

❓ Как мне вернуть товар?
[2026-07-20T07:16:50.084399] request_started: {
  "query": "Как мне вернуть товар?",
  "search_type": "hybrid"
}
[2026-07-20T07:16:51.746632] retrieval_completed: {
  "chunks_found": 3,
  "latency_ms": 1662.12,
  "top_chunk": "Возврат товара:\nДля возврата товара необходимо име..."
}
[2026-07-20T07:16:53.654626] request_completed: {
  "answer_preview": "Для возврата товара необходимо иметь при себе чек и паспорт. Срок возврата составляет 14 дней с моме...",
  "answer_length": 238,
  "retrieval_ms": 1662.12,
  "generation_ms": 1907.73,
  "total_ms": 3569.85
}
🤖 Для возврата товара необходимо иметь при себе чек и паспорт. Срок возврата составляет 14 дней с моме...


❓ Сколько стоит доставка в Казань?
[2026-07-20T07:16:53.654931] request_started: {
  "query": "Сколько стоит доставка в Казань?",
  "search_type": "hybrid"
}
[2026-07-20T07:16:54.379340] retrieval_completed: {
  "chunks_found": 3,
  "latency_ms": 724.32,
  "top_chunk": "Дост